# 🏥 Insurance RAG — Cross-Document Query Comparison (with Reranking)
**HDFC Ergo Optima Secure  vs  Care Insurance Supreme**

Both PDFs are extracted, chunked, and loaded into separate ChromaDB collections.
Every query runs against both simultaneously. We use a **two-stage retrieval pipeline**:
1. **Dense Retrieval (Bi-Encoder):** Quickly fetches the top $K$ candidate chunks.
2. **Reranking (Cross-Encoder):** Scores the exact relationship between the query and each candidate to surface the most contextually relevant chunks to the very top.

---

## 0 · Setup

In [1]:
!pip install pymupdf chromadb sentence-transformers --quiet
!pip install langchain_text_splitters langchain_chroma langchain_huggingface --quiet
print('✅ Dependencies installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [2]:
import sys, os, json, textwrap, time
from pathlib import Path

# ── If cloned from GitHub ─────────────────────────────────────────────────
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

# ── If cloned from GitHub ─────────────────────────────────────────────────
!git clone https://{token}@github.com/falcon978/Insurance-RAG
%cd Insurance-RAG

sys.path.insert(0, 'src')

from insurance_rag.pipeline import ExtractionPipeline
print('✅ Package imported')

Cloning into 'Insurance-RAG'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 54 (delta 16), reused 47 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 2.28 MiB | 31.99 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/Insurance-RAG
✅ Package imported


In [ ]:
import importlib

# Import the modules
from insurance_rag import pipeline, extractor, reconstructor, chunker, cleaner, models, patterns

# Reload the modules to reflect any code changes
importlib.reload(patterns)
importlib.reload(models)
importlib.reload(pipeline)
importlib.reload(extractor)
importlib.reload(reconstructor)
importlib.reload(chunker)
importlib.reload(cleaner)
# Import the specific classes/functions after reload
from insurance_rag.pipeline import ExtractionPipeline
from insurance_rag.extractor import PDFExtractor
from insurance_rag.reconstructor import SectionReconstructor
from insurance_rag.chunker import HierarchicalChunker
from insurance_rag.cleaner import clean_block_text

print("Modules reloaded successfully!")

---
## 1 · Download Both PDFs

In [3]:
import urllib.request

PDFS = {
    "optima_secure": {
        "name"  : "HDFC Ergo Optima Secure",
        "url"   : (
            "https://customer-portal-assets.hdfcergo.com/assets/v2/docs/"
            "default-source/downloads/policy-wordings/health/"
            "optima-secure-revision/optima-secure-revision-pw-647504209314.pdf"
        ),
        "path"  : "hdfc_optima_secure.pdf",
    },
    "care_supreme": {
        "name"  : "Care Insurance Supreme",
        "url"   : (
            "https://cms.careinsurance.com/cms/public/uploads/download_center/"
            "care-supreme---policy-terms-&-conditions-(effective-from-19-march-2025).pdf"
            "?rv=0.86869200%201775054695"
        ),
        "path"  : "care_supreme.pdf",
    },
}

for key, info in PDFS.items():
    if not Path(info['path']).exists():
        print(f"Downloading {info['name']} …")
        try:
            req = urllib.request.Request(
                info['url'],
                headers={'User-Agent': 'Mozilla/5.0'}
            )
            with urllib.request.urlopen(req, timeout=60) as r:
                Path(info['path']).write_bytes(r.read())
            size = Path(info['path']).stat().st_size // 1024
            print(f"  ✅ {info['path']}  ({size} KB)")
        except Exception as e:
            print(f"  ❌ Failed: {e}")
            print(f"  → Upload {info['path']} manually using the cell below")
    else:
        print(f"✅ {info['path']} already exists")

  ✅ hdfc_optima_secure.pdf  (610 KB)
  ✅ care_supreme.pdf  (1809 KB)


In [ ]:
# ── Manual upload fallback (run only if download failed) ──────────────────
# from google.colab import files
# uploaded = files.upload()
# Rename the uploaded files to match PDFS paths above if needed

---
## 2 · Extract & Chunk Both PDFs

In [4]:
CHUNK_SIZE = 1200
OVERLAP    = 150

results = {}

for key, info in PDFS.items():
    if not Path(info['path']).exists():
        print(f"⚠️  {info['path']} not found — skipping")
        continue

    print(f"\n{'='*60}")
    print(f"Extracting & Indexing: {info['name']}")
    print('='*60)

    # UPDATED: Pass a unique collection_name to the pipeline
    result = ExtractionPipeline(
        pdf_path        = info['path'],
        chunk_size      = CHUNK_SIZE,
        chunk_overlap   = OVERLAP,
        collection_name = f"insurance_{key}" # <-- Separates the databases!
    ).run()

    results[key] = result
    print(f"Stats: {result.stats}")


Extracting & Indexing: HDFC Ergo Optima Secure
Phase 1/4 — Extracting raw blocks …
           53 pages | 1875 blocks | 0 TOC entries
Phase 2/4 — Reconstructing sections …
           264 sections found
Phase 3/4 — Chunking for RAG (Hierarchical) …
           268 RAG chunks created
Phase 4/4 — Indexing into ChromaDB (./chroma_data) …
Initializing embedding model (this may take a moment)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Connecting to local Chroma database at './chroma_data'...
Adding 268 chunks to the database in 3 batches...
  Processed batch 1/3
  Processed batch 2/3
  Processed batch 3/3
Indexing complete!

✅  Pipeline Complete in 333.84s — 268 chunks successfully indexed!
Stats: {'total_blocks': 1875, 'total_sections': 264, 'total_chunks': 268, 'avg_chunk_chars': 535, 'avg_token_estimate': 133, 'indexed_to_db': True, 'database_path': './chroma_data', 'elapsed_seconds': 333.84}

Extracting & Indexing: Care Insurance Supreme
Phase 1/4 — Extracting raw blocks …
           83 pages | 1654 blocks | 99 TOC entries
Phase 2/4 — Reconstructing sections …
           194 sections found
Phase 3/4 — Chunking for RAG (Hierarchical) …
           220 RAG chunks created
Phase 4/4 — Indexing into ChromaDB (./chroma_data) …
Initializing embedding model (this may take a moment)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Connecting to local Chroma database at './chroma_data'...
Adding 220 chunks to the database in 3 batches...
  Processed batch 1/3
  Processed batch 2/3
  Processed batch 3/3
Indexing complete!

✅  Pipeline Complete in 268.88s — 220 chunks successfully indexed!
Stats: {'total_blocks': 1654, 'total_sections': 194, 'total_chunks': 220, 'avg_chunk_chars': 572, 'avg_token_estimate': 143, 'indexed_to_db': True, 'database_path': './chroma_data', 'elapsed_seconds': 268.88}


---
## 3 · Load Models & Index into ChromaDB

In [17]:
import chromadb
from chromadb.utils import embedding_functions
from sentence_transformers import CrossEncoder

print("Loading Cross-Encoder model for reranking...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("Connecting to the persistent ChromaDB created by the pipeline...")
# Connect to the persistent directory where Phase 4 saved the data
client = chromadb.PersistentClient(path="./chroma_data")

# We MUST tell Chroma to use the exact same embedding model that indexer.py used
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-large-en-v1.5"
)

collections = {}

for key in PDFS.keys():
    col_name = f"insurance_{key}"

    # Just GET the collection, do not upsert!
    collection = client.get_collection(
        name=col_name,
        embedding_function=emb_fn
    )
    collections[key] = collection
    print(f"✅ Connected to {col_name} -> {collection.count()} chunks found")

print("\n✅ Both collections and models ready")

Loading Cross-Encoder model for reranking...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Connecting to the persistent ChromaDB created by the pipeline...
✅ Connected to insurance_optima_secure -> 268 chunks found
✅ Connected to insurance_care_supreme -> 220 chunks found

✅ Both collections and models ready


---
## 4 · Query Interface with Two-Stage Retrieval

In [18]:
def retrieve(query: str, n: int = 3, top_k: int = 20, section_filter: str = None, apply_rerank: bool = True) -> dict:
    """
    Run query against both collections using two-stage retrieval.
    Stage 1: Fetch `top_k` chunks using fast vector distance.
    Stage 2: Rerank those chunks using Cross-Encoder and return top `n`.

    Returns {"hdfc": [...], "care": [...]}
    Each result: (text, metadata, vector_score, rerank_score)
    """
    output = {}

    for key, collection in collections.items():
        # --- STAGE 1: DENSE RETRIEVAL ---
        res = collection.query(
            query_texts = [query],
            n_results   = top_k,
            include     = ['documents', 'metadatas', 'distances'],
        )
        docs      = res['documents'][0]
        metadatas = res['metadatas'][0]
        distances = res['distances'][0]

        hits = list(zip(docs, metadatas, distances))

        # Optional post-filter by section keyword
        if section_filter:
            hits = [
                (d, m, dist) for d, m, dist in hits
                if section_filter.lower() in m['section'].lower()
                or section_filter.lower() in m.get('sub_section','').lower()
            ]

        # --- STAGE 2: CROSS-ENCODER RERANKING ---
        if apply_rerank and hits:
            # Create pairs of (Query, Document Text) for the Cross-Encoder
            pairs = [[query, doc] for doc, _, _ in hits]
            rerank_scores = cross_encoder.predict(pairs)

            # Zip everything together: (doc, meta, dist, rerank_score)
            scored_hits = [(hits[i][0], hits[i][1], hits[i][2], float(rerank_scores[i])) for i in range(len(hits))]

            # Sort descending by the new rerank score (higher is better)
            scored_hits.sort(key=lambda x: x[3], reverse=True)
            output[key] = scored_hits[:n]

        else:
            # Fallback if no reranking applied
            output[key] = [(d, m, dist, None) for d, m, dist in hits[:n]]

    return output


def show(query: str, n: int = 3, top_k: int = 20, section_filter: str = None, apply_rerank: bool = True,
         text_preview: int = 400):
    """
    Pretty-print side-by-side results for both insurers.
    """
    print(f"\n{'▓'*70}")
    print(f"  QUERY: {query}")
    print(f"  PIPELINE: Fetched top {top_k} -> Reranked to top {n}" if apply_rerank else f"  PIPELINE: Vector Search Only (top {n})")
    if section_filter:
        print(f"  FILTER: section contains '{section_filter}'")
    print(f"{'▓'*70}")

    raw = retrieve(query, n=n, top_k=top_k, section_filter=section_filter, apply_rerank=apply_rerank)

    for key, hits in raw.items():
        insurer = PDFS[key]['name']
        print(f"\n{'─'*70}")
        print(f"  📄 {insurer}")
        print(f"{'─'*70}")

        if not hits:
            print("  No results matched the filter.")
            continue

        for rank, (doc, meta, dist, rerank_score) in enumerate(hits, 1):
            v_score = round(1 - dist, 3)
            r_score_str = f"{rerank_score:.2f}" if rerank_score is not None else "N/A"
            section = meta.get('section', '')[:40]
            heading = meta.get('heading', '')[:60]
            pages   = f"p.{meta.get('page_start')}–{meta.get('page_end')}"
            preview = doc[:text_preview].replace('\n', ' ')
            if len(doc) > text_preview:
                preview += ' …'

            print(f"\n  [{rank}] Rerank Score: {r_score_str} (Vec: {v_score})  |  {pages}")
            print(f"       Section : {section}")
            print(f"       Heading : {heading}")
            print(f"       Text    : {preview}")

    print()


def show_full(query: str, rank: int = 1, insurer_key: str = 'hdfc',
              n: int = 5, top_k: int = 20):
    """
    Show the complete text of a specific result.
    Useful when the preview is too short.
    """
    raw  = retrieve(query, n=n, top_k=top_k)
    hits = raw.get(insurer_key, [])

    if rank > len(hits):
        print(f"Only {len(hits)} results available")
        return

    doc, meta, dist, rerank_score = hits[rank - 1]
    r_score_str = f"{rerank_score:.2f}" if rerank_score is not None else "N/A"
    v_score_str = f"{round(1-dist, 3)}"

    print(f"\n{'─'*70}")
    print(f"  {PDFS[insurer_key]['name']}  |  Rank {rank}")
    print(f"  Scores  : Rerank: {r_score_str} | Vector: {v_score_str}")
    print(f"  Section : {meta.get('section','')}")
    print(f"  Heading : {meta.get('heading','')}")
    print(f"  Pages   : {meta.get('page_start')}–{meta.get('page_end')}")
    print(f"{'─'*70}")
    print(doc)


print("✅ Query functions ready")
print()
print("Functions available:")
print("  show(query)                          — reranked top 3 results from both insurers")
print("  show(query, apply_rerank=False)      — standard vector search (no reranking)")
print("  show(query, section_filter='EXCL')   — filter by section keyword before reranking")
print("  show_full(query, rank=1, insurer_key='hdfc')  — full chunk text")

✅ Query functions ready

Functions available:
  show(query)                          — reranked top 3 results from both insurers
  show(query, apply_rerank=False)      — standard vector search (no reranking)
  show(query, section_filter='EXCL')   — filter by section keyword before reranking
  show_full(query, rank=1, insurer_key='hdfc')  — full chunk text


---
## 5 · Run Your Queries
Edit the query string in any cell below and run it.  
Add as many cells as you need — `show()` is all you need.

In [12]:
# ── Your query here ───────────────────────────────────────────────────────
show("Does this policy cover robotic surgery?", n=5, top_k=25)


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  QUERY: Does this policy cover robotic surgery?
  PIPELINE: Fetched top 25 -> Reranked to top 5
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

──────────────────────────────────────────────────────────────────────
  📄 HDFC Ergo Optima Secure
──────────────────────────────────────────────────────────────────────

  [1] Rerank Score: -6.55 (Vec: 0.21)  |  p.21–21
       Section : 2.  Optional Covers
       Heading : A.Global Health Cover (Emergency Treatments Only) is applica
       Text    : [2.  Optional Covers] [A.Global Health Cover (Emergency Treatments Only) is applicable subject to followingterms and conditions] ions i.  Our maximum liability in a Policy Year for claims under this cover shall not exceed the Base  Sum Insured and Plus Benefit (if available). ii. Section B-2.7 (Aggregate Deductible) will not be applicable for any claim under this cover. However, a Per Claim Deduct …

  

In [20]:
# ── Your query here ───────────────────────────────────────────────────────
show_full("Are modern treatment methods or advanced technology methods covered?", rank=4, insurer_key="optima_secure", top_k=25)


──────────────────────────────────────────────────────────────────────
  HDFC Ergo Optima Secure  |  Rank 4
  Scores  : Rerank: -9.16 | Vector: 0.16
  Section : SECTION A. DEFINITIONS
  Heading : Def. 43.Unproven/Experimental Treatment means
  Pages   : 8–8
──────────────────────────────────────────────────────────────────────
[SECTION A. DEFINITIONS] [Def. 43.Unproven/Experimental Treatment means]
s the treatment including drug experimental
therapy which is based on established medical practice in India, is a treatment experimental or 
unproven.


In [9]:
# ── With section filter ───────────────────────────────────────────────────
show("What is the waiting period for pre-existing diseases?", section_filter="waiting")


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  QUERY: What is the waiting period for pre-existing diseases?
  PIPELINE: Fetched top 20 -> Reranked to top 3
  FILTER: section contains 'waiting'
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

──────────────────────────────────────────────────────────────────────
  📄 HDFC Ergo Optima Secure
──────────────────────────────────────────────────────────────────────

  [1] Rerank Score: 7.07 (Vec: 0.484)  |  p.25–25
       Section : 2.  Accommodation Expenses
       Heading : 2.12PED waiting period modification
       Text    : [2.  Accommodation Expenses] [2.12PED waiting period modification] on On availing this option, Pre-existing Disease Waiting Period shall stand modified and will be as  stipulated in the Policy Schedule. All other terms and Conditions of the Policy shall remain  unaltered. This optional cover is allowed to be opted at channel level only and only at the time of  policy inc

In [8]:
# ── Compare Vector vs Reranker Output ─────────────────────────────────────
# Notice how apply_rerank=False might pull irrelevant text that simply shares keywords

q = "Are maternity expenses covered?"
print("\n--- WITHOUT RERANKING ---")
show(q, apply_rerank=False, n=1)
print("\n--- WITH RERANKING ---")
show(q, apply_rerank=True, n=1)


--- WITHOUT RERANKING ---

▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  QUERY: Are maternity expenses covered?
  PIPELINE: Vector Search Only (top 1)
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

──────────────────────────────────────────────────────────────────────
  📄 HDFC Ergo Optima Secure
──────────────────────────────────────────────────────────────────────

  [1] Rerank Score: N/A (Vec: 0.428)  |  p.32–32
       Section : 2.  Standard Exclusions
       Heading : o.Maternity: Code - Excl18:
       Text    : [2.  Standard Exclusions] [o.Maternity: Code - Excl18:] 8: i.  Medical treatment expenses traceable to childbirth (including complicated deliveries and  caesarean sections incurred during hospitalization) except ectopic pregnancy; ii. Expenses towards miscarriage (unless due to an accident) and lawful medical termination of pregnancy during the Policy Period.

───────────────────────────────────────────────────────────────

---
## 6 · Retrieval Quality Diagnostics
Run these after your queries to understand what's working and what isn't.

In [14]:
# ── Score distribution for a query ───────────────────────────────────────
def score_distribution(query: str, n: int = 10, top_k: int = 30):
    raw = retrieve(query, n=n, top_k=top_k, apply_rerank=True)
    print(f'Query: "{query}"\n')
    for key, hits in raw.items():
        print(f"{PDFS[key]['name']}:")
        for rank, (_, meta, dist, rerank_score) in enumerate(hits, 1):
            # Normalize cross-encoder logit loosely for visual bar mapping (approx map -10 to +10 range)
            norm_score = max(0, min(1, (rerank_score + 5) / 15)) if rerank_score else 0
            bar   = '█' * int(norm_score * 20)
            heading = meta.get('heading', '')[:40]
            r_str = f"{rerank_score:+.2f}" if rerank_score is not None else "N/A"
            v_str = f"{round(1-dist,2)}"
            print(f"  [{rank}] R:{r_str:<6} (V:{v_str:<4}) | {bar:<20} {heading}")
        print()

score_distribution("Are modern treatment methods or advanced technology methods covered?")

Query: "Are modern treatment methods or advanced technology methods covered?"

HDFC Ergo Optima Secure:
  [1] R:-7.87  (V:0.17) |                      Def. 11.Day Care Treatment means
  [2] R:-8.40  (V:0.12) |                      n.Sterility and Infertility: Code - Excl
  [3] R:-8.70  (V:0.16) |                      2.10.Global Health Cover (Emergency & Pl
  [4] R:-9.16  (V:0.16) |                      Def. 43.Unproven/Experimental Treatment 
  [5] R:-9.42  (V:0.14) |                      B.Specific Exclusions applicable to Glob
  [6] R:-9.77  (V:0.2 ) |                      Def. 6.AYUSH Treatment
  [7] R:-9.80  (V:0.13) |                      2.1.Emergency Air Ambulance
  [8] R:-9.98  (V:0.17) |                      A.Global Health Cover (Emergency Treatme
  [9] R:-9.99  (V:0.29) |                      3. 
Specific Exclusions:
  [10] R:-10.07 (V:0.15) |                      Def. 13.Dental Treatment means

Care Insurance Supreme:
  [1] R:+4.40  (V:0.38) | ████████████         (iii) Ad

In [11]:
# ── Check which sections are being retrieved for a query ──────────────────
def section_coverage(query: str, n: int = 10, top_k: int = 20):
    raw = retrieve(query, n=n, top_k=top_k, apply_rerank=True)
    print(f'Query: "{query}"\n')
    for key, hits in raw.items():
        print(f"{PDFS[key]['name']}:")
        for _, meta, dist, rerank_score in hits:
            r_str = f"{rerank_score:+.2f}" if rerank_score is not None else "N/A"
            section = meta.get('section', '')[:35]
            heading = meta.get('heading', '')[:35]
            print(f"  {r_str:>6}  [{section}]  {heading}")
        print()

section_coverage("How is day care treatment handled?")

Query: "How is day care treatment handled?"

HDFC Ergo Optima Secure:
   +3.62  [SECTION A. DEFINITIONS]  Def. 11.Day Care Treatment means
   -0.50  [SECTION A. DEFINITIONS]  Def. 10.Day Care Centre means
   -1.07  [1.  Base Coverage]  iv.
   -1.21  [SECTION A. DEFINITIONS]  Def. 33.OPD Treatment means
   -2.58  [1.  Claims Procedure]  Type of ClaimPrescribed Time limit
   -2.89  [Table of Contents]  Operating Clause
   -3.04  [SECTION A. DEFINITIONS]  Def. 3.AYUSH Hospital
   -3.73  [1.  Base Coverage]  1.2.Home Health Care
   -3.89  [SECTION A. DEFINITIONS]  Def. 18.Hospitalmeans
   -5.36  [2.  Standard Exclusions]  i.

Care Insurance Supreme:
   +4.04  [2. Definitions]  2.1.10. Day Care Treatment
   +2.15  [3. Benefits Covered Under The Polic]  (ii)Benefit: Day Care Treatment:
   +1.60  [2. Definitions]  2.1.9. Day Care Centre
   -0.87  [2. Definitions]  2.1.34. OPD Treatment
   -1.85  [3. Benefits Covered Under The Polic]  General Conditions Applicable To Al
   -3.15  [3. Benefits 

In [ ]:
# ── Chunk size stats per insurer ──────────────────────────────────────────
# Quick sanity check that chunks are well-sized

for key, result in results.items():
    chunks     = result.chunks
    char_counts = [len(c.text) for c in chunks]
    print(f"{PDFS[key]['name']}")
    print(f"  Total chunks : {len(chunks)}")
    print(f"  Min chars    : {min(char_counts)}")
    print(f"  Max chars    : {max(char_counts)}")
    print(f"  Mean chars   : {sum(char_counts)//len(char_counts)}")
    print(f"  Mean tokens~ : {sum(char_counts)//(len(char_counts)*4)}")
    print()

---
## 7 · LLM Generation (The Legal Assistant)
This stage takes the highly specific chunks retrieved by the Cross-Encoder and feeds them into an LLM.
The LLM is strictly constrained by a system prompt to prevent hallucinations and to explicitly advise the user on what to ask the insurer if a clause is "silent" (missing).

In [ ]:
# Install the necessary LangChain and LLM packages
!pip install langchain langchain-google-genai --quiet
print('✅ LLM dependencies installed')

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from IPython.display import display, Markdown

# 1. Setup your API Key (Get a free one at https://aistudio.google.com/)
from google.colab import userdata
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    # Fallback if not using Colab's secret manager
    GEMINI_API_KEY = input("Enter your Google Gemini API Key: ")

# Initialize the LLM (Temperature 0 ensures it doesn't get "creative" with legal text)
llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0, api_key=GEMINI_API_KEY)

# 2. Define the Strict System Guardrails
system_prompt = """
You are an expert Insurance Policy Analyst AI. You are provided with a user query and the most relevant extracted text chunks from two different insurance policies.

YOUR RULES:
1. EXPLICIT MATCH: If the document explicitly mentions the queried term, explain the coverage, sub-limits, or exclusions based ONLY on the text provided.
2. ABSENCE OF TERM: If the document DOES NOT explicitly mention the queried term, DO NOT guess, hallucinate, or assume coverage. State clearly that it is not explicitly mentioned.
3. CONTEXTUAL REASONING: If the exact term is missing, look at the provided "catch-all" clauses (e.g., "Modern Treatments", "Experimental Treatments", "Definitions"). Explain how these clauses *might* apply to the user's query.
4. THE CALL TO ACTION: If there is ambiguity or an absence of explicit mention, you MUST explicitly advise the user to contact the insurance company for written confirmation.
5. ADVOCACY: Provide 2-3 specific, highly targeted questions the user should ask the insurer, using the exact definitions and clause numbers found in the text.
6. FORMATTING: Use Markdown. Create a clear header for each insurer. Be concise and professional.
"""

# Create the LangChain Prompt Template
prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "USER QUERY: {query}\n\n--- HDFC ERGO OPTIMA SECURE CONTEXT ---\n{hdfc_context}\n\n--- CARE SUPREME CONTEXT ---\n{care_context}")
])

# Create the Generation Chain
analysis_chain = prompt_template | llm

print("✅ LLM Pipeline and Guardrails Ready")

In [ ]:
def generate_policy_comparison(query: str, top_k: int = 30, n: int = 3):
    """
    1. Runs the Two-Stage Retrieval (Bi-Encoder -> Cross-Encoder).
    2. Formats the retrieved chunks into a context string.
    3. Feeds the context and query to the LLM for legal analysis.
    """
    print(f"🔍 Retrieving context for: '{query}'...")

    # 1. Fetch the best chunks using our RAG pipeline
    raw_results = retrieve(query, n=n, top_k=top_k, apply_rerank=True)

    # 2. Helper function to format chunks into readable text for the LLM
    def format_context(hits):
        if not hits:
            return "No relevant clauses found in the document."

        context_str = ""
        for rank, (doc, meta, dist, rerank_score) in enumerate(hits, 1):
            context_str += f"[Rank {rank} | Section: {meta.get('section', 'Unknown')}]\n"
            context_str += f"Heading: {meta.get('heading', 'Unknown')}\n"
            context_str += f"Text: {doc}\n\n"
        return context_str

    # Format context for both insurers
    hdfc_text = format_context(raw_results.get('optima_secure', []))
    care_text = format_context(raw_results.get('care_supreme', []))

    print("🧠 Analyzing legal text with LLM...\n")
    print("="*80)

    # 3. Generate the response
    response = analysis_chain.invoke({
        "query": query,
        "hdfc_context": hdfc_text,
        "care_context": care_text
    })

    # Display nicely in the notebook
    # Check if response.content is a list of dictionaries and extract text
    if isinstance(response.content, list) and all(isinstance(item, dict) and 'text' in item for item in response.content):
        markdown_text = "".join([item['text'] for item in response.content])
        display(Markdown(markdown_text))
    else:
        display(Markdown(response.content))

print("✅ Generation function ready. Run queries using generate_policy_comparison('your question')")

In [ ]:
# Test the end-to-end pipeline!
query = "Are there any limits, caps, or proportionate deductions applied to room rent or ICU charges?"
generate_policy_comparison(query)